# 1D Black-Scholes Equations Solver Exploration

The goal of this notebook is to provide a preliminary understanding of the problem statement, the object and current existing methods.

## Interactive 3D Render (Zoom/Rotate)
Use this Plotly figure to zoom, pan, and rotate the BS surface interactively.

In [6]:
# Interactive Black-Scholes 3D surface with Plotly
import numpy as np
from scipy.stats import norm
import plotly.graph_objects as go

K = 1000.0
r = 0.05
sigma = 0.2
T = 100.0
S_max = 220.0

S = np.linspace(1e-6, S_max, 160)
t = np.linspace(0.0, T, 140)
S_grid, t_grid = np.meshgrid(S, t)
tau = T - t_grid

sqrt_tau = np.sqrt(np.maximum(tau, 1e-12))
d1 = (np.log(S_grid / K) + (r + 0.5 * sigma**2) * tau) / (sigma * sqrt_tau)
d2 = d1 - sigma * sqrt_tau
V = S_grid * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)

payoff = np.maximum(S - K, 0.0)
V[-1, :] = payoff

fig = go.Figure()
fig.add_trace(go.Surface(
    x=S_grid,
    y=t_grid,
    z=V,
    colorscale='Viridis',
    opacity=0.92,
    colorbar=dict(title='V(S,t)')
))

fig.add_trace(go.Scatter3d(
    x=S,
    y=np.full_like(S, T),
    z=payoff,
    mode='lines',
    line=dict(color='crimson', width=6),
    name='Terminal payoff t=T'
))

fig.update_layout(
    title='Interactive European Call Surface V(S,t) under Black-Scholes',
    scene=dict(
        xaxis_title='Underlying price S',
        yaxis_title='Time t',
        zaxis_title='Option value V'
    ),
    width=950,
    height=700,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()


## Black-Scholes Setup and Derivation via Portfolio Process

### Assumptions
- The underlying follows geometric Brownian motion under the physical measure: \
  \`dS_t = \mu S_t dt + \sigma S_t dW_t\`.
- Volatility \`\sigma\` and risk-free rate \`r\` are constants.
- No dividends (or continuous dividend yield is set to zero).
- Frictionless market: no transaction costs, no taxes, continuous trading, infinitely divisible assets.
- No arbitrage and unlimited borrowing/lending at rate \`r\`.
- European payoff depends only on \`S_T\` at maturity \`T\`.

### Notation and Definitions
- \`S_t\`: underlying price at time \`t\`.
- \`V(S,t)\`: option value as a function of \`(S,t)\`.
- \`K\`: strike, \`T\`: maturity, \`\tau = T-t\`: time-to-maturity.
- \`W_t\`: standard Brownian motion.
- Terminal condition (European call): \`V(S,T)=\max(S-K,0)\`.

### Derivation from a Self-Financing Hedged Portfolio
Apply Itô's lemma to \`V(S_t,t)\`:
\[
dV = \left(V_t + \tfrac12 \sigma^2 S^2 V_{SS} + \mu S V_S\right)dt + \sigma S V_S dW_t.
\]
Construct a portfolio with one option and short \`\Delta\` shares:
\[
\Pi = V - \Delta S.
\]
Choose \`\Delta = V_S\` so the diffusion term cancels:
\[
d\Pi = \left(V_t + \tfrac12 \sigma^2 S^2 V_{SS}\right)dt.
\]
This portfolio is locally riskless, so by no-arbitrage it must earn \`r\`:
\[
d\Pi = r\Pi dt = r(V - SV_S)dt.
\]
Equating drifts gives the Black-Scholes PDE:
\[
V_t + \tfrac12 \sigma^2 S^2 V_{SS} + rSV_S - rV = 0, \quad (S,t)\in(0,\infty)\times[0,T).
\]
With terminal condition \`V(S,T)=\max(S-K,0)\` (call), the unique no-arbitrage solution is:
\[
V(S,t)=S\,N(d_1)-Ke^{-r(T-t)}N(d_2),
\]
\[
d_1=\frac{\ln(S/K)+(r+\tfrac12\sigma^2)(T-t)}{\sigma\sqrt{T-t}}, \qquad d_2=d_1-\sigma\sqrt{T-t}.
\]
